<a href="https://colab.research.google.com/github/Rizm10/Asthma-risk-precautions-system-final-year-project-/blob/notebooks/Asthma_ML_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports for collection
!pip install requests
import requests
import pandas as pd
import time

# **Retrieving Daily Environmental data from Open-Mateo API**

In [ ]:
# NOTE: This cell is for DATA COLLECTION ONLY.
# Model evaluation uses the saved DAILY CSVs to ensure reproducibility.

import requests
import pandas as pd
import time

# -----------------------------
# CONFIG
# -----------------------------
lat, lon = 51.5072, -0.1276   # London coordinates
year = 2023

# -----------------------------
# 1️⃣ AIR QUALITY (Hourly → Daily)
# -----------------------------
air_frames = []

for month in range(1, 13):
    start = f"{year}-{month:02d}-01"
    end_month = month + 1 if month < 12 else 1
    end_year = year if month < 12 else year + 1
    end = f"{end_year}-{end_month:02d}-01"

    air_url = (
        f"https://air-quality-api.open-meteo.com/v1/air-quality?"
        f"latitude={lat}&longitude={lon}"
        f"&hourly=pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,ozone,sulphur_dioxide"
        f"&start_date={start}&end_date={end}"
    )

    res = requests.get(air_url)
    res.raise_for_status()
    data = res.json()

    df = pd.DataFrame(data["hourly"])
    air_frames.append(df)
    print(f"Fetched AIR {start} → {end} ({len(df)} rows)")
    time.sleep(1)

air_hourly = pd.concat(air_frames, ignore_index=True)
air_hourly["time"] = pd.to_datetime(air_hourly["time"])
air_hourly["date"] = air_hourly["time"].dt.date

air_daily = (
    air_hourly
    .drop(columns=["time"])
    .groupby("date", as_index=False)
    .mean(numeric_only=True)
)

air_daily.to_csv(f"air_quality_daily_{year}.csv", index=False)
print(f"\n✅ Air Quality (daily): {len(air_daily)} records saved\n")

# -----------------------------
# 2️⃣ POLLEN (Hourly → Daily)
# -----------------------------
pollen_frames = []

for month in range(1, 13):
    start = f"{year}-{month:02d}-01"
    end_month = month + 1 if month < 12 else 1
    end_year = year if month < 12 else year + 1
    end = f"{end_year}-{end_month:02d}-01"

    pollen_url = (
        f"https://air-quality-api.open-meteo.com/v1/air-quality?"
        f"latitude={lat}&longitude={lon}"
        f"&hourly=alder_pollen,birch_pollen,grass_pollen,olive_pollen,ragweed_pollen"
        f"&start_date={start}&end_date={end}"
    )

    res = requests.get(pollen_url)
    res.raise_for_status()
    data = res.json()

    df = pd.DataFrame(data["hourly"])
    pollen_frames.append(df)
    print(f"Fetched POLLEN {start} → {end} ({len(df)} rows)")
    time.sleep(1)

pollen_hourly = pd.concat(pollen_frames, ignore_index=True)
pollen_hourly = pollen_hourly.fillna(0)  # off-season → 0
pollen_hourly["time"] = pd.to_datetime(pollen_hourly["time"])
pollen_hourly["date"] = pollen_hourly["time"].dt.date

pollen_daily = (
    pollen_hourly
    .drop(columns=["time"])
    .groupby("date", as_index=False)
    .mean(numeric_only=True)
)

pollen_daily.to_csv(f"pollen_daily_{year}.csv", index=False)
print(f"\n✅ Pollen (daily): {len(pollen_daily)} records saved\n")

# -----------------------------
# 3️⃣ WEATHER (Already Daily)
# -----------------------------
weather_url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={lat}&longitude={lon}"
    f"&start_date={year}-01-01&end_date={year+1}-01-01"
    f"&daily=temperature_2m_max,temperature_2m_min,"
    f"precipitation_sum,relative_humidity_2m_mean"
)

res = requests.get(weather_url)
res.raise_for_status()
data = res.json()

weather_daily = pd.DataFrame(data["daily"])
weather_daily.rename(columns={"time": "date"}, inplace=True)

# Ensure same date type as the others (python date objects)
weather_daily["date"] = pd.to_datetime(weather_daily["date"]).dt.date

weather_daily.to_csv("weather_daily_2024.csv", index=False)
print(f"\n✅ Weather (daily): {len(weather_daily)} records saved\n")

Fetched AIR 2023-01-01 → 2023-02-01 (768 rows)
Fetched AIR 2023-02-01 → 2023-03-01 (696 rows)
Fetched AIR 2023-03-01 → 2023-04-01 (768 rows)
Fetched AIR 2023-04-01 → 2023-05-01 (744 rows)
Fetched AIR 2023-05-01 → 2023-06-01 (768 rows)
Fetched AIR 2023-06-01 → 2023-07-01 (744 rows)
Fetched AIR 2023-07-01 → 2023-08-01 (768 rows)
Fetched AIR 2023-08-01 → 2023-09-01 (768 rows)
Fetched AIR 2023-09-01 → 2023-10-01 (744 rows)
Fetched AIR 2023-10-01 → 2023-11-01 (768 rows)
Fetched AIR 2023-11-01 → 2023-12-01 (744 rows)
Fetched AIR 2023-12-01 → 2024-01-01 (768 rows)

✅ Air Quality (daily): 366 records saved

Fetched POLLEN 2023-01-01 → 2023-02-01 (768 rows)
Fetched POLLEN 2023-02-01 → 2023-03-01 (696 rows)
Fetched POLLEN 2023-03-01 → 2023-04-01 (768 rows)
Fetched POLLEN 2023-04-01 → 2023-05-01 (744 rows)
Fetched POLLEN 2023-05-01 → 2023-06-01 (768 rows)
Fetched POLLEN 2023-06-01 → 2023-07-01 (744 rows)
Fetched POLLEN 2023-07-01 → 2023-08-01 (768 rows)
Fetched POLLEN 2023-08-01 → 2023-09-01 (768

/tmp/ipython-input-852362881.py:82: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pollen_hourly = pd.concat(pollen_frames, ignore_index=True)



✅ Pollen (daily): 366 records saved


✅ Weather (daily): 366 records saved



# **  Air Quality data   AQA**
A total of 366 records of hourly data were extracted from [Open Mateo Air Quality API](https://open-meteo.com/en/docs/air-quality-api?timezone=GMT&time_mode=time_interval&start_date=2023-01-13&end_date=2024-01-01). from the year 2024 data The following was collected:

*   time
*   pm10
*   pm2_5
*   carbon_monoxide
*   nitrogen_dioxide  
*   ozone
*   sulphur_dioxide










In [ ]:
aqa_df = pd.read_csv(f"air_quality_daily_{year}.csv")
print(aqa_df.head())
print(len(aqa_df), "total records collected")

         date       pm10     pm2_5  carbon_monoxide  nitrogen_dioxide  \
0  2023-01-01  14.420833  7.762500       152.416667         12.979167   
1  2023-01-02  12.575000  9.500000       201.375000         29.929167   
2  2023-01-03  10.566667  7.137500       170.541667         21.458333   
3  2023-01-04  10.854167  6.483333       121.791667         11.741667   
4  2023-01-05  13.408333  7.900000       137.625000         17.025000   

       ozone  sulphur_dioxide  
0  60.541667         3.462500  
1  32.375000         4.854167  
2  45.625000         3.400000  
3  60.000000         3.041667  
4  51.000000         3.629167  
366 total records collected


In [ ]:
aqa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              366 non-null    object 
 1   pm10              366 non-null    float64
 2   pm2_5             366 non-null    float64
 3   carbon_monoxide   366 non-null    float64
 4   nitrogen_dioxide  366 non-null    float64
 5   ozone             366 non-null    float64
 6   sulphur_dioxide   366 non-null    float64
dtypes: float64(6), object(1)
memory usage: 20.1+ KB


# **Pollen data**
A total of 366 records of hourly data were extracted from [Open Mateo Pollen API](https://open-meteo.com/en/docs/air-quality-api?timezone=GMT&time_mode=time_interval&start_date=2023-01-13&end_date=2024-01-01).filtered from the year 2023  The following was collected:

*   time
*   alder_pollen
*   birch_pollen
*   grass_pollen
*   olive_pollen  
*   rageweed_pollen



In [ ]:
pollen_df = pd.read_csv(f"pollen_daily_{year}.csv")
print(len(pollen_df))
print(pollen_df.head())
print(len(pollen_df), "total records collected")

366
         date  alder_pollen  birch_pollen  grass_pollen  olive_pollen  \
0  2023-01-01           0.0           0.0           0.0           0.0   
1  2023-01-02           0.0           0.0           0.0           0.0   
2  2023-01-03           0.0           0.0           0.0           0.0   
3  2023-01-04           0.0           0.0           0.0           0.0   
4  2023-01-05           0.0           0.0           0.0           0.0   

   ragweed_pollen  
0             0.0  
1             0.0  
2             0.0  
3             0.0  
4             0.0  
366 total records collected


In [ ]:
pollen_df.describe()

,alder_pollen,birch_pollen,grass_pollen,olive_pollen,ragweed_pollen
count,9072.000000,9072.000000,9072.000000,9072.000000,9072.000000
mean,0.831085,2.041931,3.410439,0.005688,0.015983
std,8.289053,12.866812,9.991304,0.095232,0.161490
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.900000,0.000000,0.000000
max,240.300000,205.000000,107.300000,5.500000,4.000000


Due to pollen being highly seasonal majority of rows outside of spring/summer will likely be zero. It has been decided for accuracy of data to be left alone as it simulates accurate real life results

# **Weather data**
A total of 366 records of daily data were extracted from [Open Mateo Air Quality API](https://open-meteo.com/en/docs/air-quality-api?timezone=GMT&time_mode=time_interval&start_date=2023-01-13&end_date=2024-01-01). from the year 2024 The following was collected:

*   time
*   temperature_2m
*   relative_humidity_2m
*   precipitation
*   dew_point_2m      
*   wind_speed_10m



In [ ]:
weather_df = pd.read_csv("weather_daily_2024.csv")
print(len(weather_df), "total records collected")
df.head()

366 total records collected


,time,alder_pollen,birch_pollen,grass_pollen,olive_pollen,ragweed_pollen
0,2023-12-01T00:00,0.0,0.0,0.0,0.0,0.0
1,2023-12-01T01:00,0.0,0.0,0.0,0.0,0.0
2,2023-12-01T02:00,0.0,0.0,0.0,0.0,0.0
3,2023-12-01T03:00,0.0,0.0,0.0,0.0,0.0
4,2023-12-01T04:00,0.0,0.0,0.0,0.0,0.0


In [ ]:
weather_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       366 non-null    object 
 1   temperature_2m_max         366 non-null    float64
 2   temperature_2m_min         366 non-null    float64
 3   precipitation_sum          366 non-null    float64
 4   relative_humidity_2m_mean  366 non-null    int64  
dtypes: float64(3), int64(1), object(1)
memory usage: 14.4+ KB


# **Merging environmental data**
in this section Environmental data will be merged to form env_df


In [ ]:
env_df = pd.merge(aqa_df, pollen_df, on="date", how="inner")
env_df = pd.merge(env_df, weather_df, on="date", how="inner")
print(env_df.head())

         date       pm10     pm2_5  carbon_monoxide  nitrogen_dioxide  \
0  2023-01-01  14.420833  7.762500       152.416667         12.979167   
1  2023-01-02  12.575000  9.500000       201.375000         29.929167   
2  2023-01-03  10.566667  7.137500       170.541667         21.458333   
3  2023-01-04  10.854167  6.483333       121.791667         11.741667   
4  2023-01-05  13.408333  7.900000       137.625000         17.025000   

       ozone  sulphur_dioxide  alder_pollen  birch_pollen  grass_pollen  \
0  60.541667         3.462500           0.0           0.0           0.0   
1  32.375000         4.854167           0.0           0.0           0.0   
2  45.625000         3.400000           0.0           0.0           0.0   
3  60.000000         3.041667           0.0           0.0           0.0   
4  51.000000         3.629167           0.0           0.0           0.0   

   olive_pollen  ragweed_pollen  temperature_2m_max  temperature_2m_min  \
0           0.0             0.0    

In [ ]:
env_df.to_csv("env_df.csv", index=False)

In [ ]:
env_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       366 non-null    object 
 1   pm10                       366 non-null    float64
 2   pm2_5                      366 non-null    float64
 3   carbon_monoxide            366 non-null    float64
 4   nitrogen_dioxide           366 non-null    float64
 5   ozone                      366 non-null    float64
 6   sulphur_dioxide            366 non-null    float64
 7   alder_pollen               366 non-null    float64
 8   birch_pollen               366 non-null    float64
 9   grass_pollen               366 non-null    float64
 10  olive_pollen               366 non-null    float64
 11  ragweed_pollen             366 non-null    float64
 12  temperature_2m_max         366 non-null    float64
 13  temperature_2m_min         366 non-null    float64

# **Asthma incident data**

A total of 217528 total  records were extract from [Public Health England Fingertips API](https://fingertips.phe.org.uk/profile/guidance/supporting-information/api). Which then was filtered for the year 2023/24. This resulted in 33578 records being retained for analysis from the following Asthma related indicators.

*   Hospital admissions for asthma (under 19 years) (9774 records)
*   Asthma: QOF prevalence (8221)
*   Patients with Asthma: review in the last 12 months (denominator incl PCAs) (7723)
*   Patients with Asthma (6-19 yrs): Second-hand smoking status recorded in the last 12 months (denominator incl. PCAs)	(7703)
*   Emergency hospital admissions for asthma in adults (aged 19 years and over)	(157)

The following code snippet demonstrates how data for the selected asthma indicators was retrieved using the Fingertips API




In [ ]:
import requests
import pandas as pd
!pip install fingertips-py

import fingertips_py as ftp
import pandas as pd

# Indicator ID for "Hospital admissions for asthma (under 19 years)"
indicator_id = 90810

# Fetch data for all available geographies
df = ftp.get_data_for_indicator_at_all_available_geographies(indicator_id)

print(df.head())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
   Indicator ID                                   Indicator Name Parent Code  \
0         90810  Hospital admissions for asthma (under 19 years)         NaN   
1         90810  Hospital admissions for asthma (under 19 years)         NaN   
2         90810  Hospital admissions for asthma (under 19 years)         NaN   
3         90810  Hospital admissions for asthma (under 19 years)         NaN   
4         90810  Hospital admissions for asthma (under 19 years)         NaN   

  Parent Name  Area Code Area Name Area Type   Sex       Age  \
0         NaN  E92000001   England   England  Male  0-18 yrs   
1         NaN  E92000001   England   England  Male  0-18 yrs   
2         NaN  E92000001   England   England  Male  0-18 yrs   
3         NaN  E92000001   England   England  Male  0-18 yrs   
4         NaN  E92000001   England   England  Male  0-18 yrs   

                                       Category Type  

In [ ]:
cols = [
    "Indicator ID", "Indicator Name", "Area Code", "Area Name",
    "Sex", "Age", "Time period", "Value"
]
df_clean = df[cols]
print(df_clean.head())

   Indicator ID                                   Indicator Name  Area Code  \
0         90810  Hospital admissions for asthma (under 19 years)  E92000001   
1         90810  Hospital admissions for asthma (under 19 years)  E92000001   
2         90810  Hospital admissions for asthma (under 19 years)  E92000001   
3         90810  Hospital admissions for asthma (under 19 years)  E92000001   
4         90810  Hospital admissions for asthma (under 19 years)  E92000001   

  Area Name   Sex       Age Time period       Value  
0   England  Male  0-18 yrs     2013/14  232.464199  
1   England  Male  0-18 yrs     2013/14  384.660000  
2   England  Male  0-18 yrs     2013/14  275.740000  
3   England  Male  0-18 yrs     2013/14  235.300000  
4   England  Male  0-18 yrs     2013/14  237.970000  


In [ ]:
import fingertips_py as ftp
import pandas as pd
import time

indicator_ids = [90933, 92780, 93790, 93791, 93644, 93573, 93594, 93595, 90810]
all_data = []

for ind in indicator_ids:
    print(f"Fetching {ind} ...")
    try:
        df_temp = ftp.get_data_for_indicator_at_all_available_geographies(ind)
        df_temp["Indicator ID"] = ind
        all_data.append(df_temp)
        time.sleep(1)
    except Exception as e:
        print(f"⚠️ Failed for {ind}: {e}")

df = pd.concat(all_data, ignore_index=True)
print(df[["Indicator ID", "Indicator Name", "Area Name", "Time period", "Value"]].head())

Fetching 90933 ...
Fetching 92780 ...
⚠️ Failed for 92780: 'NoneType' object is not iterable
Fetching 93790 ...
Fetching 93791 ...
Fetching 93644 ...
Fetching 93573 ...
Fetching 93594 ...
⚠️ Failed for 93594: 'NoneType' object is not iterable
Fetching 93595 ...
⚠️ Failed for 93595: 'NoneType' object is not iterable
Fetching 90810 ...
   Indicator ID          Indicator Name Area Name Time period     Value
0         90933  Asthma: QOF prevalence   England     2020/21  6.375014
1         90933  Asthma: QOF prevalence   England     2020/21  6.550504
2         90933  Asthma: QOF prevalence   England     2020/21  6.422434
3         90933  Asthma: QOF prevalence   England     2020/21  6.225627
4         90933  Asthma: QOF prevalence   England     2020/21  5.969579


In [ ]:
df.to_csv("asthma_full_dataset.csv", index=False)

# **Asthma dataset 2023/24 and cleaning**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234546 entries, 0 to 234545
Data columns (total 27 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   Indicator ID                              234546 non-null  int64  
 1   Indicator Name                            234546 non-null  object 
 2   Parent Code                               230909 non-null  object 
 3   Parent Name                               230909 non-null  object 
 4   Area Code                                 234546 non-null  object 
 5   Area Name                                 234546 non-null  object 
 6   Area Type                                 234546 non-null  object 
 7   Sex                                       234546 non-null  object 
 8   Age                                       234546 non-null  object 
 9   Category Type                             6742 non-null    object 
 10  Category            

In [ ]:
df['Time period'].unique()

array(['2020/21', '2021/22', '2022/23', '2023/24', '2024/25', '2001 - 03',
       '2002 - 04', '2003 - 05', '2004 - 06', '2005 - 07', '2006 - 08',
       '2007 - 09', '2008 - 10', '2009 - 11', '2010 - 12', '2011 - 13',
       '2012 - 14', '2013 - 15', '2014 - 16', '2015 - 17', '2016 - 18',
       '2017 - 19', '2018 - 20', '2019 - 21', '2020 - 22', '2021 - 23',
       '2022 - 24', '2011', '2012', '2013', '2014', '2015', '2016',
       '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024',
       '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008',
       '2009', '2010', '2013/14', '2014/15', '2015/16', '2016/17',
       '2017/18', '2018/19', '2019/20'], dtype=object)

In [ ]:
df_2023_24 = df[df["Time period"].astype(str).str.contains("2023/24", na=False)]
print(df_2023_24.head())
print(len(df_2023_24), "total records collected for 2023/24")

     Indicator ID          Indicator Name Parent Code Parent Name  Area Code  \
705         90933  Asthma: QOF prevalence         NaN         NaN  E92000001   
706         90933  Asthma: QOF prevalence         NaN         NaN  E92000001   
707         90933  Asthma: QOF prevalence         NaN         NaN  E92000001   
708         90933  Asthma: QOF prevalence         NaN         NaN  E92000001   
709         90933  Asthma: QOF prevalence         NaN         NaN  E92000001   

    Area Name Area Type      Sex     Age  \
705   England   England  Persons  6+ yrs   
706   England   England  Persons  6+ yrs   
707   England   England  Persons  6+ yrs   
708   England   England  Persons  6+ yrs   
709   England   England  Persons  6+ yrs   

                                         Category Type  ...      Count  \
705                                                NaN  ...  3886879.0   
706  County & UA deprivation deciles in England (IM...  ...   389788.0   
707  County & UA deprivation dec

In [ ]:
df_2023_24["Indicator Name"].value_counts()

,count
Indicator Name,
Hospital admissions for asthma (under 19 years),9774
Asthma: QOF prevalence,8158
Patients with Asthma: review in the last 12 months (denominator incl. PCAs),7620
Patients with Asthma (6-19 yrs): Second-hand smoking status recorded in the last 12 months (denominator incl. PCAs),7620
Emergency hospital admissions for asthma in adults (aged 19 years and over),157


In [ ]:
df_2023_24.to_csv("asthma_2023_24.csv", index=False)

In [ ]:
drop_columns =  []
keep_columns = [ "Value" , "Indicator Name"]
for col in df_2023_24.columns:
    if col not in keep_columns:
        drop_columns.append(col)
clean_asthma_df = df_2023_24.drop(columns = drop_columns)



# **Justification of dropping columns**

All columns Except Value and indicator name have been dropped due to being noise. Indicator name has been kept for stratified sampling

In [ ]:
clean_asthma_df.head()

,Indicator Name,Value
705,Asthma: QOF prevalence,6.525525
706,Asthma: QOF prevalence,6.546683
707,Asthma: QOF prevalence,6.500385
708,Asthma: QOF prevalence,6.325449
709,Asthma: QOF prevalence,5.976655


In [ ]:
clean_asthma_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33329 entries, 705 to 234545
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Indicator Name  33329 non-null  object 
 1   Value           31931 non-null  float64
dtypes: float64(1), object(1)
memory usage: 781.1+ KB


In [ ]:
clean_asthma_df.to_csv("clean_asthma_df.csv", index=False) #Saved pre sampling for repreducibility

# **Stratify Sampling justification**
Due to high incidents records of 32180  relative to training data too better reflect real world rarity of asthmetic Excerbations  [ERS Publications](https://https://publications.ersnet.org/content/erj/58/6/2100413) approximated 8-12% of patients with Asthma experience >= exacerbations per year.
Therefore for modellling purposes, the incidences will be sampled at 11% to represent the upper-midpoint of this established range.




In [ ]:
asthma_stratified = clean_asthma_df.groupby('Indicator Name', group_keys=False)\
    .apply(lambda x: x.sample(frac=0.11, random_state=42)) # random_state used for reproducibility

/tmp/ipython-input-1849536907.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.11, random_state=42)) # random_state used for reproducibility


In [ ]:
asthma_stratified.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3665 entries, 30755 to 45340
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Indicator Name  3665 non-null   object 
 1   Value           3501 non-null   float64
dtypes: float64(1), object(1)
memory usage: 85.9+ KB


In [ ]:
asthma_stratified = asthma_stratified.drop(columns = ["Indicator Name"]) # unnecessary now as data is stratified

In [ ]:
asthma_stratified.to_csv("asthma_stratified.csv", index= False)

In [ ]:
asthma_stratified.describe()

,Value
count,3501.000000
mean,77.667881
std,68.233866
min,0.000000
25%,9.078170
50%,71.513350
75%,90.481361
max,559.159025


**Train the ML & interpretation**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # Changed back to LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# -----------------------------
# Target (Admissions)
# -----------------------------
asthma_ml = asthma_stratified.copy()
y = asthma_ml["Value"].astype(float)

# -----------------------------
# Features (Environmental)
# -----------------------------
# Use MEAN environmental exposure (yearly) repeated for each admissions row
env_mean = env_df.drop(columns=["date"]).mean().values.reshape(1, -1)

X = pd.DataFrame(
    np.repeat(env_mean, repeats=len(y), axis=0),
    columns=env_df.drop(columns=["date"]).columns
)

# Drop rows with NaN values from both X and y to ensure alignment
combined_df = pd.concat([X, y.rename('target')], axis=1)
combined_df.dropna(inplace=True)
X = combined_df.drop('target', axis=1)
y = combined_df['target']

# -----------------------------
# Train / Test
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression() # Changed back to LinearRegression
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R²:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R²: -0.04356309708331296
RMSE: 1.0394715654639486


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest Results")
print("RMSE:", rmse)
print("R²:", r2)

Random Forest Results
RMSE: 1.04019022108465
R²: -0.04500656462893815


In [ ]:
import pandas as pd

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance

,0
pm10,0.0
pm2_5,0.0
carbon_monoxide,0.0
nitrogen_dioxide,0.0
ozone,0.0
sulphur_dioxide,0.0
alder_pollen,0.0
birch_pollen,0.0
grass_pollen,0.0
olive_pollen,0.0


# **Observed performance**

Both logestic regression (linear) and random forests (non-linear) performed extremely poorly, achieving RMSEs below 2 and negative R² scores. and feature importance values equal to 0 across all environmental variables.

This indicates no meaningful predictive signal was learn from the environmental features. This suggests that the limitation lies in the temporal and semantic mismatch between ecological exposure data and publicly available health outcomes, rather than model choice or algorithm configuration.


# **Results interpretation**


As mentioned in Observed performances the machine learning approach has demonstrated consistent poor results across both linear and non linear models, demonstrating no meaningfull patterns between environmental data and asthma admissions not attributed to algorith choice.

Additionally, attempt to aggregate high resolution environmental features to low resolution hospital admissions data did not recover predictive signal, this suggest information loss during temporal alignment.

Public asthma admisions data reflects population level delays and multi factor outcomes which are not directly responsive to environmental variations which affect onset symtoms of excerbations. As a result, environmental factors likely act as risk modifiers rather than standolne drivers of hospital admissions.
